# 结果分析

In [1]:
import torch
import json
from pathlib import Path

from config.config import PATH_CFG, ID_TO_ACTION, RLInferenceConfig, STOP_ACTION_ID
from model.actor_network import ActorNetwork
from utils.chem import smiles_to_graph
from tokenizer.tokenizer import LabelTokenizer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

checkpoint_path = Path("ckpt/pretrain_20260426_gat_uncertainty/best_model.pt")

# 1. 加载分词器
tokenizer = LabelTokenizer(str(PATH_CFG.VOCAB_FILE))
vocab_size = tokenizer.vocab_size

# 2. 构建模型
model = ActorNetwork(vocab_size=vocab_size).to(device)

# 3. 加载训练好的模型
# checkpoint_path = PATH_CFG.CKPT_BEST_MODEL_FILE
if not checkpoint_path.exists():
    print(f"Error: Checkpoint file not found at {checkpoint_path}")

print(f"Loading checkpoint from {checkpoint_path}")
ckpt = torch.load(checkpoint_path, map_location=device, weights_only=True)
model.load_state_dict(ckpt['model'])
model.eval()



/root/autodl-tmp/enzretro/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading checkpoint from ckpt/pretrain_20260426_gat_uncertainty/best_model.pt


ActorNetwork(
  (encoder): GATEncoder(
    (dropout): Dropout(p=0.1, inplace=False)
    (input_proj): Linear(in_features=78, out_features=512, bias=True)
    (convs): ModuleList(
      (0-3): 4 x GATConv(512, 128, heads=4)
    )
    (norms): ModuleList(
      (0-3): 4 x LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    )
    (acts): ModuleList(
      (0-3): 4 x GELU(approximate='none')
    )
  )
  (state_proj): Linear(in_features=512, out_features=512, bias=True)
  (state_tracker): StateTracker(
    (history_encoder): HistoryEncoder(
      (action_emb): Embedding(7, 512, padding_idx=6)
      (atom_emb): Embedding(201, 512, padding_idx=200)
      (label_token_emb): Embedding(137, 512, padding_idx=0)
      (fusion): Sequential(
        (0): Linear(in_features=2048, out_features=512, bias=True)
        (1): ReLU()
        (2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      )
    )
    (gru): GRU(512, 512, batch_first=True)
    (fusion_proj): Sequential(
      (0): Lin

In [2]:
test_smiles = "CCOc1ccc(Oc2ncnc3c2cnn3C2CCN(C(=O)OC3CCCC3)CC2)c(F)c1"

# 5. 转换为图表示
x, edge_index, edge_attr = smiles_to_graph(test_smiles)
if x is None:
    print("Error: Failed to parse SMILES")

# 6. 准备输入数据
# 添加batch维度
x = x.to(device)  # [N, 79]
edge_index = edge_index.to(device)  # [2, E]
edge_attr = edge_attr.to(device)  # [E, 12]
batch = torch.zeros(x.size(0), dtype=torch.long, device=device)  # [N]

# 7. 调用模型进行推理
encoder_kwargs = {
    'x': x,
    'edge_index': edge_index.squeeze(0),  # GAT encoder expects [2, E]
    'batch': batch
}

encoder_kwargs

{'x': tensor([[1., 0., 0.,  ..., 0., 0., 0.],
         [1., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 1.,  ..., 0., 0., 0.],
         ...,
         [1., 0., 0.,  ..., 0., 1., 1.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [1., 0., 0.,  ..., 0., 1., 1.]], device='cuda:0'),
 'edge_index': tensor([[ 0,  1,  1,  2,  2,  3,  3,  4,  4,  5,  5,  6,  6,  7,  7,  8,  8,  9,
           9, 10, 10, 11, 11, 12, 12, 13, 13, 14, 14, 15, 15, 16, 16, 17, 17, 18,
          18, 19, 19, 20, 20, 21, 21, 22, 21, 23, 23, 24, 24, 25, 25, 26, 26, 27,
          27, 28, 20, 29, 29, 30,  6, 31, 31, 32, 31, 33, 33,  3, 13,  8, 30, 17,
          16, 12, 28, 24],
         [ 1,  0,  2,  1,  3,  2,  4,  3,  5,  4,  6,  5,  7,  6,  8,  7,  9,  8,
          10,  9, 11, 10, 12, 11, 13, 12, 14, 13, 15, 14, 16, 15, 17, 16, 18, 17,
          19, 18, 20, 19, 21, 20, 22, 21, 23, 21, 24, 23, 25, 24, 26, 25, 27, 26,
          28, 27, 29, 20, 30, 29, 31,  6, 32, 31, 33, 31,  3, 33,  8, 13, 17, 30,
          12, 16, 2

In [3]:
x.shape, edge_index.shape, batch.shape

(torch.Size([34, 78]), torch.Size([2, 76]), torch.Size([34]))

In [4]:
with torch.no_grad():
    edit_steps = model.generate(**encoder_kwargs)

edit_steps


[EditStep(action_type=tensor([3], device='cuda:0'), src_idx=tensor([32], device='cuda:0'), tgt_idx=tensor([0], device='cuda:0'), label_tokens=tensor([[ 1, 23, 47,  3, 40, 46, 40, 58,  3, 40, 40, 46, 40, 46, 40,  3, 21, 40,
          46, 40, 46, 40, 40,  3,  3, 40, 46, 40, 46, 40, 46, 40]],
        device='cuda:0')),
 EditStep(action_type=tensor([3], device='cuda:0'), src_idx=tensor([24], device='cuda:0'), tgt_idx=tensor([0], device='cuda:0'), label_tokens=tensor([[ 1, 23, 47,  3,  3,  3, 40, 46, 40, 58,  3, 40, 58,  3,  3,  3, 21, 40,
          58,  3, 40, 40, 46, 40, 46, 40, 40, 46, 22, 40, 46, 40]],
        device='cuda:0')),
 EditStep(action_type=tensor([5], device='cuda:0'), src_idx=tensor([19], device='cuda:0'), tgt_idx=tensor([5], device='cuda:0'), label_tokens=tensor([[ 1, 23, 47,  3,  3,  3, 47,  3,  3, 47,  3,  3, 47,  3,  3, 47,  3,  3,
          47,  3, 47,  3, 46,  3,  3, 47,  3,  3, 47,  3,  3, 47]],
        device='cuda:0'))]

In [7]:
# 检查第一步的预测是否正确
if edit_steps:
    first_step = edit_steps[0]
    pred_action = first_step.action_type.item()
    pred_src = first_step.src_idx.item()
    pred_tgt = first_step.tgt_idx.item()
    
    # 解码标签序列
    label_tokens = first_step.label_tokens.squeeze(0).tolist()
    pred_label = tokenizer.decode(label_tokens, skip_special_tokens=True)

    print(f"  Action: {ID_TO_ACTION[pred_action]} (id={pred_action})")
    print(f"  Source atom: {pred_src}")
    print(f"  Target atom: {pred_tgt}")
    print(f"  Label: {pred_label}")
    print()

  Action: AttachGroup (id=3)
  Source atom: 32
  Target atom: 0
  Label: *Cl



In [2]:
import torch
from collections import Counter
from data.dataset import USPTO50KDataset

from tokenizer.tokenizer import LabelTokenizer
from config.config import config as cfg
from config.config import ACTION_TO_ID

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 加载 tokenizer 和模型结构
tokenizer  = LabelTokenizer(vocab_file=str(cfg.path.VOCAB_FILE))

# 1. Action 分布
ds = USPTO50KDataset(str(cfg.path.RL_TEST_DATA_FILE), tokenizer)
action_counter = Counter()
for sample in ds.samples:
    for edit in sample["edits"]:
        action_counter[edit["action_id"]] += 1
        
id_to_action = {v: k for k, v in ACTION_TO_ID.items()}
total = sum(action_counter.values())
print("\n=== Action 分布 ===")
for aid, cnt in sorted(action_counter.items()):
    print(f"  [{aid}] {id_to_action.get(aid,'?'):<25} {cnt:>6}  {cnt/total*100:.2f}%")



/root/autodl-tmp/enzretro/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 5006 samples, skipped 1

=== Action 分布 ===
  [0] DeleteBond                  3704  25.01%
  [1] ChangeBond                   588  3.97%
  [2] ChangeAtom                   180  1.22%
  [3] AttachGroup                 5334  36.01%
  [4] Terminate                   5006  33.80%


In [1]:
import torch
from model.actor_pretrainer import ActorPretrainer
from tokenizer.tokenizer import LabelTokenizer
from config.config import MODEL_CONFIG, PATH_CONFIG

C  = MODEL_CONFIG
PC = PATH_CONFIG

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 加载 tokenizer 和模型结构
tokenizer  = LabelTokenizer(vocab_file=PC["vocab_file"])
vocab_size = tokenizer.get_vocab_size()

print(vocab_size)

/root/autodl-tmp/enzretro/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


137


In [9]:
from data.dataset import USPTO50KDataset
json_path = "dataset/uspto50k/processed/uspto50k_test_output.json"
dataset = USPTO50KDataset(json_path, tokenizer)
dataset[0]

NameError: name 'torch' is not defined

In [3]:
import torch
from model.actor_pretrainer import ActorPretrainer
from tokenizer.tokenizer import LabelTokenizer
from config.config import MODEL_CONFIG, PATH_CONFIG

C  = MODEL_CONFIG
PC = PATH_CONFIG

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 加载 tokenizer 和模型结构
tokenizer  = LabelTokenizer(vocab_file=PC["vocab_file"])
vocab_size = tokenizer.get_vocab_size()
model      = ActorPretrainer(vocab_size=vocab_size).to(device)

# 加载权重
ckpt = torch.load(PC.get("ckpt_path", "best_model.pt"), map_location=device)
model.load_state_dict(ckpt["model"])
model.eval()  # ← 关键，关闭 dropout / batchnorm

print(f"✅ Loaded best model  loss={ckpt['loss']:.4f}  @ {ckpt['epoch_tag']}")


✅ Loaded best model  loss=4.2105  @ ep98


/tmp/ipykernel_7701/913752287.py:17: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(PC.get("ckpt_path", "best_model.pt"), map_location=device)


In [2]:
from torch_geometric.utils import to_dense_batch

@torch.no_grad()   # ← 关键，推理不需要梯度
def predict(batch):
    batch = batch.to(device)

    # 图编码
    node_embeddings, graph_embedding = model.graph_encoder(
        batch.x, batch.edge_index, batch.batch)
    graph_state   = model.state_proj(graph_embedding)
    decoder_state = model.state_tracker(batch.history_actions, graph_state)

    # 预测 action
    action_logits = model.action_predictor(decoder_state)
    pred_action   = action_logits.argmax(dim=-1)          # [B]

    # 预测 src / tgt
    dense_nodes, node_mask_bool = to_dense_batch(node_embeddings, batch.batch)
    src_logits, tgt_logits = model.pointer_network(
        decoder_state, dense_nodes, pred_action,
        node_mask=~node_mask_bool)
    pred_src = src_logits.argmax(dim=-1)                  # [B]
    pred_tgt = tgt_logits.argmax(dim=-1)                  # [B]

    # 预测 label（自回归解码）
    pred_label = greedy_decode_label(decoder_state, pred_action)

    return pred_action, pred_src, pred_tgt, pred_label

@torch.no_grad()
def greedy_decode_label(decoder_state, pred_action,
                        max_len=32):
    B      = decoder_state.size(0)
    # 用 BOS token 作为起始输入
    tokens = torch.full((B, 1), tokenizer.bos_token_id,
                        dtype=torch.long, device=device)

    for _ in range(max_len):
        logits     = model.label_decoder(decoder_state, pred_action, tokens)
        next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)  # [B, 1]
        tokens     = torch.cat([tokens, next_token], dim=-1)

        # 所有样本都生成了 EOS 则提前停止
        if (next_token.squeeze(-1) == tokenizer.eos_token_id).all():
            break

    return tokens[:, 1:]   # 去掉 BOS


In [3]:
from torch_geometric.loader import DataLoader
from data.ssr_graph_pretrain_dataset import SSRGraphDataset

dataset    = SSRGraphDataset(
    json_path=PC["test_data"], tokenizer=tokenizer,
    max_seq_len=C["max_seq_len"], max_hist_len=C["max_hist_len"],
)
dataloader = DataLoader(dataset, batch_size=32, shuffle=False)

correct_action = correct_src = correct_tgt = total = 0

for batch in dataloader:
    pred_action, pred_src, pred_tgt, _ = predict(batch)

    target_action = batch.target_action.squeeze(-1).to(device)
    target_src    = batch.target_src.squeeze(-1).to(device)
    target_tgt    = batch.target_tgt.squeeze(-1).to(device)

    correct_action += (pred_action == target_action).sum().item()
    correct_src    += (pred_src    == target_src).sum().item()
    correct_tgt    += (pred_tgt    == target_tgt).sum().item()
    total          += target_action.size(0)

print(f"Action Acc : {correct_action / total:.4f}")
print(f"Src    Acc : {correct_src    / total:.4f}")
print(f"Tgt    Acc : {correct_tgt    / total:.4f}")


Action Acc : 0.8748
Src    Acc : 0.2286
Tgt    Acc : 0.0892


In [5]:
import torch 
from torch_geometric.utils import to_dense_batch

@torch.no_grad()
def predict_oracle(batch):
    """用真实 action/src 作为输入，排除级联误差"""
    batch = batch.to(device)

    node_embeddings, graph_embedding = model.graph_encoder(
        batch.x, batch.edge_index, batch.batch)
    graph_state   = model.state_proj(graph_embedding)
    decoder_state = model.state_tracker(batch.history_actions, graph_state)

    target_action = batch.target_action.squeeze(-1)
    target_src    = batch.target_src.squeeze(-1)

    dense_nodes, node_mask_bool = to_dense_batch(node_embeddings, batch.batch)

    # 用真实 action 预测 src
    src_logits, tgt_logits = model.pointer_network(
        decoder_state, dense_nodes, target_action,
        target_src_idx=target_src,                  # ← 用真实 src 预测 tgt
        node_mask=~node_mask_bool)

    pred_src = src_logits.argmax(dim=-1)
    pred_tgt = tgt_logits.argmax(dim=-1)
    return pred_src, pred_tgt

from torch_geometric.loader import DataLoader
from data.ssr_graph_pretrain_dataset import SSRGraphDataset

dataset    = SSRGraphDataset(
    json_path=PC["test_data"], tokenizer=tokenizer,
    max_seq_len=C["max_seq_len"], max_hist_len=C["max_hist_len"],
)
dataloader = DataLoader(dataset, batch_size=32, shuffle=False)

correct_action = correct_src = correct_tgt = total = 0

for batch in dataloader:
    pred_src, pred_tgt = predict_oracle(batch)

    target_action = batch.target_action.squeeze(-1).to(device)
    target_src    = batch.target_src.squeeze(-1).to(device)
    target_tgt    = batch.target_tgt.squeeze(-1).to(device)

    correct_src    += (pred_src    == target_src).sum().item()
    correct_tgt    += (pred_tgt    == target_tgt).sum().item()
    total          += target_action.size(0)

print(f"Action Acc : {correct_action / total:.4f}")
print(f"Src    Acc : {correct_src    / total:.4f}")
print(f"Tgt    Acc : {correct_tgt    / total:.4f}")

Action Acc : 0.0000
Src    Acc : 0.2383
Tgt    Acc : 0.0969
